# Week 68, Day 2: ctypes and cffi

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w068d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ctypes

libc = ctypes.CDLL(None)            # the C library already loaded into this process
print(libc.abs(-7))
print(libc.strlen(b"hello"))
libc.toupper.restype = ctypes.c_int
print(chr(libc.toupper(ord("q"))))

In [ ]:
import ctypes

libc = ctypes.CDLL(None)
libc.strlen.argtypes = [ctypes.c_char_p]
libc.strlen.restype = ctypes.c_size_t
libc.atoi.argtypes = [ctypes.c_char_p]
libc.atoi.restype = ctypes.c_int
libc.strcmp.argtypes = [ctypes.c_char_p, ctypes.c_char_p]
libc.strcmp.restype = ctypes.c_int

print(libc.strlen("naïve".encode()), "bytes in UTF-8")
print(libc.atoi(b"  42 apples"), libc.atoi(b"none"))
print(libc.strcmp(b"apple", b"banana") < 0, libc.strcmp(b"same", b"same"))
try:
    libc.strlen("not bytes")
except ctypes.ArgumentError as e:
    print("ArgumentError:", str(e)[:40])

In [ ]:
import ctypes

n = ctypes.c_int(41)
n.value += 1
print(n.value, ctypes.sizeof(ctypes.c_int), ctypes.sizeof(ctypes.c_int64), ctypes.sizeof(ctypes.c_double))
print(ctypes.c_uint8(300).value, ctypes.c_int8(200).value)
print(ctypes.sizeof(ctypes.c_void_p) in (4, 8))

libc = ctypes.CDLL(None)
libc.labs.argtypes = [ctypes.c_long]
libc.labs.restype = ctypes.c_long
print(libc.labs(-123456))

In [ ]:
import ctypes

libc = ctypes.CDLL(None)
libc.strtol.argtypes = [ctypes.c_char_p, ctypes.POINTER(ctypes.c_char_p), ctypes.c_int]
libc.strtol.restype = ctypes.c_long

end = ctypes.c_char_p()
value = libc.strtol(b"255 apples", ctypes.byref(end), 10)
print(value, end.value)
print(libc.strtol(b"ff", None, 16), libc.strtol(b"777", None, 8))

numbers = (ctypes.c_int * 4)(10, 20, 30, 40)
print(list(numbers), ctypes.sizeof(numbers), numbers[2])

buffer = ctypes.create_string_buffer(10)
libc.memset(buffer, ord("*"), 3)
libc.memmove(ctypes.byref(buffer, 3), b"ok", 2)
print(buffer.value, buffer.raw)

In [ ]:
import ctypes

class Reading(ctypes.Structure):
    _fields_ = [("flag", ctypes.c_char), ("value", ctypes.c_int), ("unit", ctypes.c_char)]

class PackedReading(ctypes.Structure):
    _pack_ = 1
    _fields_ = [("flag", ctypes.c_char), ("value", ctypes.c_int), ("unit", ctypes.c_char)]

for cls in (Reading, PackedReading):
    print(cls.__name__, ctypes.sizeof(cls), [getattr(cls, name).offset for name, _ in cls._fields_])

r = Reading(b"A", 21, unit=b"C")
r.value += 1
print(r.flag, r.value, r.unit, bytes(r)[:1])

class Point(ctypes.Structure):
    _fields_ = [("x", ctypes.c_int), ("y", ctypes.c_int)]

class Line(ctypes.Structure):
    _fields_ = [("start", Point), ("end", Point)]

line = Line(Point(0, 0), Point(3, 4))
print(line.end.x, line.end.y, ctypes.sizeof(Line))

In [ ]:
import ctypes

libc = ctypes.CDLL(None)
COMPARE = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int))

calls = []
def by_value(a, b):
    calls.append(1)
    return a[0] - b[0]

compare = COMPARE(by_value)                     # keep a reference while C uses it
values = (ctypes.c_int * 6)(5, 3, 9, 1, 7, 2)
libc.qsort(values, len(values), ctypes.sizeof(ctypes.c_int), compare)
print(list(values), "after", len(calls), "calls into Python")

descending = COMPARE(lambda a, b: b[0] - a[0])
libc.qsort(values, len(values), ctypes.sizeof(ctypes.c_int), descending)
print(list(values))

> **Key idea.** ctypes calls functions in shared libraries from pure Python: load with CDLL (CDLL(None) for symbols already loaded), always set argtypes and restype, and pass strings as bytes. C types (c_int, c_double, c_char_p, c_int64 ...) wrap values in .value; sizes vary by platform (c_long, pointers), and conversions wrap silently. byref and POINTER pass pointers, (c_int * n)(...) makes arrays, create_string_buffer makes writable memory, and a pointer to a pointer reports where a parser stopped. Structure subclasses with _fields_ describe structs; fields are padded to their alignment unless _pack_ = 1, and sizeof and .offset show the real layout. CFUNCTYPE wraps Python functions as C callbacks (keep a reference). Mistakes crash rather than raise, and variadic functions are fragile; cffi takes C declarations directly and offers a compiled API mode.

### Exercise: Declaring and calling

Using the C library from `ctypes.CDLL(None)`, write `c_strlen(text)` (the length in **bytes** of a Python string encoded as UTF-8, using `strlen`), `c_atoi(text)` (using `atoi`, which reads a leading whole number and gives 0 if there isn't one) and `c_compare(a, b)` (using `strcmp`, returning `-1`, `0` or `1` by the sign of the result). Declare `argtypes` and `restype` for each function.

In [ ]:
import ctypes

libc = ctypes.CDLL(None)

def c_strlen(text):
    ___

def c_atoi(text):
    ___

def c_compare(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_calls():\n    \"calling the C library\"\n    assert c_strlen(\"hello\") == 5 and c_strlen(\"caf\u00e9\") == 5 and c_strlen(\"\") == 0\n    assert c_atoi(\"42 apples\") == 42 and c_atoi(\"-7\") == -7 and c_atoi(\"none\") == 0\n    assert c_compare(\"apple\", \"banana\") == -1 and c_compare(\"pear\", \"pear\") == 0 and c_compare(\"b\", \"a\") == 1\n    assert \"argtypes\" in source and \"restype\" in source")

### Exercise: Parsing with strtol

`strtol(text, &end, base)` reads a whole number in a given base and sets `end` to point at the first character it didn't use. Write `parse_number(text, base=10)`: call it through `ctypes` and return `(value, rest)`, where `rest` is the unused part of the text as a Python `str`; raise `ValueError(f"no number in {text!r}")` if it used nothing (the rest is the whole text). Then `parse_all(text)`: read whitespace-separated whole numbers, in any of these forms: decimal, `0x` hexadecimal or `0` octal, using base `0` (which tells `strtol` to work it out), and return the list.

In [ ]:
import ctypes

libc = ctypes.CDLL(None)

def parse_number(text, base=10):
    ___

def parse_all(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"pointers to pointers\"\n    assert parse_number(\"255 apples\") == (255, \" apples\") and parse_number(\"ff\", 16) == (255, \"\")\n    assert parse_number(\"  -12kg\") == (-12, \"kg\") and parse_number(\"101\", 2) == (5, \"\")\n    try:\n        parse_number(\"apples\")\n        assert False\n    except ValueError as e:\n        assert str(e) == \"no number in 'apples'\"\n    assert parse_all(\"10 0x1f 017 -3\") == [10, 31, 15, -3] and parse_all(\"\") == []")

### Exercise: Arrays and buffers

Write `to_c_array(values)`: a `ctypes` array of `c_int` holding the values; `from_c_array(array)`: a plain Python list; `repeat_char(char, n)`: a `bytes` object of `n` copies of a one-character string, made by filling a `create_string_buffer` with `memset` from the C library (not by Python repetition); and `join_bytes(parts)`: one `bytes` value of all the parts, copied one after another into a single buffer with `memmove` and `ctypes.byref(buffer, offset)`.

In [ ]:
import ctypes

libc = ctypes.CDLL(None)

def to_c_array(values):
    ___

def from_c_array(array):
    ___

def repeat_char(char, n):
    ___

def join_bytes(parts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_arrays():\n    \"C memory\"\n    import ctypes\n    arr = to_c_array([3, 1, 4])\n    assert isinstance(arr, ctypes.Array) and arr._type_ is ctypes.c_int and len(arr) == 3 and from_c_array(arr) == [3, 1, 4]\n    assert ctypes.sizeof(arr) == 3 * ctypes.sizeof(ctypes.c_int) and from_c_array(to_c_array([])) == []\n    assert repeat_char(\"-\", 5) == b\"-----\" and repeat_char(\"x\", 0) == b\"\"\n    assert join_bytes([b\"he\", b\"ll\", b\"o\"]) == b\"hello\" and join_bytes([]) == b\"\"\n    assert \"memset\" in source and \"memmove\" in source")

### Exercise: Struct layouts

Write `layout(fields, packed=False)`: compute a C struct's layout yourself from a list of `(name, size)` pairs, where each field's alignment equals its size: each field starts at the next offset that's a multiple of its alignment (no padding at all if `packed`), and the total size is rounded up to a multiple of the largest alignment (not if packed). Return `(offsets, size)` with `offsets` a dict. Then `matches_ctypes(fields_spec)`: build a real `ctypes.Structure` subclass from a list of `(name, ctype)` pairs and return True if `layout` with `ctypes.sizeof` of each type gives the same offsets and size as `ctypes` does.

In [ ]:
import ctypes

def layout(fields, packed=False):
    ___

def matches_ctypes(fields_spec):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_structs():\n    \"padding and alignment\"\n    assert layout([(\"a\", 1), (\"b\", 4), (\"c\", 1)]) == ({\"a\": 0, \"b\": 4, \"c\": 8}, 12)\n    assert layout([(\"a\", 1), (\"b\", 4), (\"c\", 1)], packed=True) == ({\"a\": 0, \"b\": 1, \"c\": 5}, 6)\n    assert layout([(\"b\", 4), (\"a\", 1), (\"c\", 1)]) == ({\"b\": 0, \"a\": 4, \"c\": 5}, 8)\n    assert layout([(\"x\", 8), (\"y\", 2)]) == ({\"x\": 0, \"y\": 8}, 16) and layout([]) == ({}, 0)\n    import ctypes\n    assert matches_ctypes([(\"flag\", ctypes.c_char), (\"value\", ctypes.c_int), (\"unit\", ctypes.c_char)])\n    assert matches_ctypes([(\"a\", ctypes.c_int16), (\"b\", ctypes.c_char), (\"c\", ctypes.c_int32)])")

### Exercise: Sorting with a Python callback

Write `c_sort(values, reverse=False)`: copy a list of ints into a C array, sort it with the C library's `qsort` using a comparison callback made with `ctypes.CFUNCTYPE`, and return a new sorted list (descending if `reverse`). Then `c_sort_by(values, key)`: sort by `key(value)` instead; `qsort` isn't stable (equal items can end up in any order), so break ties by the value itself. Count how many times the callback was called in the attribute `c_sort.calls` (reset on each call).

In [ ]:
import ctypes

libc = ctypes.CDLL(None)

def c_sort(values, reverse=False):
    ___

def c_sort_by(values, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_callback():\n    \"qsort and CFUNCTYPE\"\n    assert c_sort([5, 3, 9, 1]) == [1, 3, 5, 9] and c_sort([5, 3, 9, 1], reverse=True) == [9, 5, 3, 1]\n    assert c_sort.calls > 0 and c_sort([]) == [] and c_sort.calls == 0\n    assert c_sort_by([12, 7, 25, 3], key=lambda v: v % 10) == [12, 3, 25, 7]\n    assert c_sort_by([-3, 2, -1], key=abs) == [-1, 2, -3]\n    assert \"CFUNCTYPE\" in source and \"qsort\" in source")

### Bug hunt: A str where C wanted bytes

`strlen` is declared to take a C string, which `ctypes` passes as `bytes`, but this code hands it a Python `str`, so the call fails with `ArgumentError`. Encode the text to UTF-8 bytes first so this prints `5`.

In [ ]:
import ctypes

libc = ctypes.CDLL(None)
libc.strlen.argtypes = [ctypes.c_char_p]
libc.strlen.restype = ctypes.c_size_t
print(libc.strlen("hello"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"C strings are bytes\"\n    expect_output(\"5\")")

### Bug hunt: A wrapped-around value

A temperature is stored in a `c_uint8`, which holds only 0 to 255, so `-5` silently wraps round to 251 and this prints `251`. Use a signed 8-bit type, `c_int8`, so it prints `-5`.

In [ ]:
import ctypes

reading = ctypes.c_uint8(-5)
print(reading.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right C type\"\n    expect_output(\"-5\")")

### Bug hunt: Padding that wasn't expected

A file format stores each record as a `char` followed by an `int` with **no** padding (5 bytes), but this struct uses the platform's normal alignment, so it's 8 bytes long and this prints `8`. Pack it with `_pack_ = 1` so its layout matches the file and this prints `5`.

In [ ]:
import ctypes

class Record(ctypes.Structure):
    _fields_ = [("kind", ctypes.c_char), ("value", ctypes.c_int32)]

print(ctypes.sizeof(Record))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"_pack_\"\n    expect_output(\"5\")")

### Puzzle: Declaring a C function

Arrange the lines that load the C library and declare `strlen` properly before calling it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # ctypes.c_uint8(260).value
q = ___   # the result type ctypes assumes when restype isn't set: "int" or "double"
r = ___   # ctypes.sizeof of a struct with a c_char and then a c_int32, normal alignment
s = ___   # the same struct with _pack_ = 1
t = ___   # True or False: a ctypes mistake always raises a Python exception

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == \"int\" and r == 8, \"p, q and r\"\n    assert s == 5 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a Pythonic wrapper

Raw `ctypes` calls are awkward to use. Wrap part of the C library in a class that feels like Python: `CString(text)` holds a writable C buffer containing the text encoded as UTF-8, with extra capacity:

- `CString(text, capacity=None)`: a buffer of `capacity` bytes (default: the encoded length plus 1); `ValueError("capacity too small")` if the text and its terminating zero don't fit;
- `len(s)` uses `strlen`; `str(s)` decodes the buffer's value; `bytes(s)` gives the text's bytes;
- `s.upper()` changes the buffer **in place**, byte by byte with C's `toupper` (only ASCII letters change), and returns `s`;
- `s.append(more)` copies `more` onto the end with `memmove`, raising `ValueError("capacity too small")` if it wouldn't fit (leaving `s` unchanged);
- `s.find(sub)`: the position of the first occurrence of `sub` (as bytes) using `strstr` (whose result is a pointer: subtract the buffer's `ctypes.addressof`), or `-1`;
- `s == other` compares the texts with `strcmp`, for another `CString` or a `str`.

In [ ]:
import ctypes

libc = ctypes.CDLL(None)

class CString:
    def __init__(self, text, capacity=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cstring():\n    \"wrapping C in Python\"\n    s = CString(\"hello\", capacity=16)\n    assert len(s) == 5 and str(s) == \"hello\" and bytes(s) == b\"hello\"\n    assert s.upper() is s and str(s) == \"HELLO\"\n    s.append(\", world\")\n    assert str(s) == \"HELLO, world\" and len(s) == 12\n    try:\n        s.append(\"x\" * 10)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"capacity too small\"\n    assert str(s) == \"HELLO, world\"\n    assert s.find(\"world\") == 7 and s.find(\"LL\") == 2 and s.find(\"zzz\") == -1\n    assert s == \"HELLO, world\" and s == CString(\"HELLO, world\") and not (s == \"hello\")\n    try:\n        CString(\"toolong\", capacity=4)\n        assert False\n    except ValueError:\n        pass\n    assert len(CString(\"\")) == 0 and str(CString(\"caf\u00e9\").upper()) == \"CAF\u00e9\"")

## Warm-up

Three functions from the C standard library, called directly from Python: no C code of ours, no compiler. `ctypes` converted the Python values to C values, called the functions, and converted the results back.

## Loading a library and calling it

- **Load** a shared library with `ctypes.CDLL(path)`: a file path, or a name found by `ctypes.util.find_library("m")`. `CDLL(None)` gives the symbols already loaded into the running program, which on Linux and in this browser includes the C library. (On Windows, use `ctypes.WinDLL` or `ctypes.CDLL("msvcrt")`.)
- **Functions** are attributes of the library object: `libc.strlen`.
- **Declare their types** before calling: `func.argtypes = [ctypes.c_char_p]` and `func.restype = ctypes.c_size_t`. Without declarations, `ctypes` guesses: it assumes the result is a C `int`, and converts arguments by their Python type (int to `int`, bytes to `char *`). Guessing wrong gives garbage or a crash, so always declare.
- **Strings**: C strings are bytes. Pass `b"text"` (or `"text".encode()`); a `c_char_p` result comes back as `bytes`.

On your own computer, a library you've compiled yourself loads the same way:

```
import ctypes
lib = ctypes.CDLL("./libgeometry.so")       # or geometry.dll on Windows
lib.area.argtypes = [ctypes.c_double, ctypes.c_double]
lib.area.restype = ctypes.c_double
print(lib.area(3.0, 4.0))
```

## C types and conversions

`ctypes` has a class for each C type: `c_bool`, `c_char` (one byte), `c_int`, `c_uint`, `c_long`, `c_longlong`, `c_size_t`, `c_float`, `c_double`, `c_char_p` (a C string), `c_void_p` (any pointer), and fixed-width ones such as `c_int8`, `c_uint16`, `c_int32`, `c_int64`.

- `ctypes.c_int(5)` creates a C value; `.value` reads or changes it.
- `ctypes.sizeof(ctypes.c_int)` is its size in bytes. Sizes depend on the platform: `c_long` is 8 bytes on 64-bit Linux and macOS but 4 on Windows, and pointers are 4 bytes here in the browser (32-bit WebAssembly). Use the fixed-width types when the size matters.
- Conversions **don't check for overflow**: `c_uint8(300).value` is `44`, silently wrapped (the same pitfall as NumPy's small integers, Week 65).
- `c_double` arguments accept Python ints and floats; `c_int` arguments reject floats.

## Pointers, arrays and buffers

C functions often take **pointers**: to write a result back, or to receive an array.

- `ctypes.byref(value)` passes a pointer to a C value (the cheap, common way); `ctypes.pointer(value)` makes a real pointer object, and `ptr.contents` is what it points to. Declare such parameters as `ctypes.POINTER(ctypes.c_int)` and so on.
- **Arrays**: `(ctypes.c_int * 5)(3, 1, 4, 1, 5)` creates a C array of five ints; it can be indexed, iterated and passed where a pointer is expected.
- **Buffers**: C functions that write text need writable memory: `ctypes.create_string_buffer(size)` or `create_string_buffer(b"initial", size)`. `.value` is the text up to the first zero byte; `.raw` is every byte.
- A function that reports **where it stopped** takes a pointer to a pointer: `strtol(text, &end, base)` sets `end` to the first character it didn't use.

## Structs and their layout

A C `struct` becomes a subclass of `ctypes.Structure` with a `_fields_` list of `(name, type)` pairs. Instances can be created with positional or keyword values, read and written by attribute, passed by pointer to C functions, and nested.

Structs in C usually contain **padding**: each field is placed at an offset that's a multiple of its **alignment** (normally its size), and the whole struct is padded to a multiple of its largest alignment. So a `char`, an `int` and another `char` take 12 bytes, not 6. `ctypes` follows the platform's rules, so `ctypes.sizeof` and `Struct.field.offset` tell you the real layout. `_pack_ = 1` removes the padding, for formats that are packed on disk or on the wire. Getting a struct layout wrong is a common source of mysterious bugs, which is why the `struct` module (Week 23) and `ctypes` let you check sizes.

## Callbacks, safety and cffi

**Callbacks**: some C functions call a function you give them, such as `qsort`, which sorts an array using your comparison function. `ctypes.CFUNCTYPE(restype, *argtypes)` makes a function-pointer type; wrapping a Python function in it gives something C can call. Keep a reference to the wrapped callback for as long as C might call it, or it may be garbage collected and the program crash.

**Safety**: `ctypes` gives you all of C's power, including its ways to crash: wrong argument types, a too-small buffer, a dangling pointer, or a callback that's been freed can corrupt memory or kill the interpreter with no Python traceback. Always declare `argtypes` and `restype`, size buffers carefully, and keep owners of memory alive. **Variadic** functions such as `printf` are especially fragile (and don't work reliably in WebAssembly at all).

**cffi** (a free third-party library, `pip install cffi`) is the main alternative. You paste the C declarations, and it works out the types:

```
from cffi import FFI
ffi = FFI()
ffi.cdef("size_t strlen(const char *s); int abs(int n);")
libc = ffi.dlopen(None)
print(libc.strlen(b"hello"), libc.abs(-3))
```

Its **API mode** compiles a small C wrapper with your compiler, which is faster and checks types against the real headers. `ctypes` needs nothing installed; `cffi` is more convenient for large C APIs. PyPy prefers `cffi`.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, find a shared library on your system (on Linux, `ctypes.util.find_library("m")` finds the maths library) and call one of its functions with `ctypes`, declaring its types from the documentation (`man 3 cos`). Tomorrow: Cython, which compiles Python-like code to C.